# Lohnt sich PCA? Dimensionsreduktion genauer untersuchen

**Autor:** jd · **Datum:** 2026-10-26 · Issue #213

**Anlass:** Feedback aus dem Statusmeeting – zeigen, *wie sehr* sich die Reduktion der Dimensionen lohnt. Bisher gab es nur fünf PCA-Punkte in der Trade-off-Kurve (#155).

**Leitfragen:**

1. Ab wie vielen Komponenten ist die Qualität erreicht (Knie der Kurve)?
2. Wie hängt das mit der erklärten Varianz zusammen?
3. Was gewinnt man praktisch (Merkmale, Trainings- und Vorhersagezeit)?
4. Was steckt in den Komponenten (Ladungen über der Wellenlänge)?
5. Lohnt sich PCA zusätzlich auf den 10 Sentinel-2-Bändern?
6. Ist das modellabhängig (kombinierte SVM, hierarchische SVM, logistische Regression, kNN)?

**Vorgehen:** 80/20-Split, CV auf dem Trainingsteil, MLflow-Experiment `band-reduction`. PCA wird in jedem Trainingsfold auf den standardisierten Spektren gefittet (`PreprocessingConfig(spectral_pca_components=k, use_spectral_standard_scale=True)`). Jede Variante bekommt ihr **eigenes Tuning**: Der beste `gamma`-Wert der RBF-SVM hängt von der Merkmalszahl ab, darum ist das Raster breit.

**Regeln (vor dem Blick auf die Ergebnisse festgelegt):**

- **Knie:** wenigste Komponenten mit höchstens `REDUCTION_MAX_CV_LOSS` (0,01) Verlust in der CV-BAcc kombiniert gegenüber der Referenz ohne PCA (wie in #155).
- **Validierung:** höchstens eine, nur für die **kombinierte SVM** (Projekt-Referenz). Ihr Knie wird empfohlen und validiert, wenn es um mindestens `MIN_CV_GAIN` (0,005) besser ist als die Referenz oder die Zeit für Training + Vorhersage mindestens halbiert (`PCA_MIN_SPEEDUP` = 2; `pca_recommendation`). Geprüft wird zuerst Hyperion, dann Sentinel-2; validiert wird höchstens die erste empfohlene Variante. Messungen spart PCA nie, weil sie alle Bänder als Eingabe braucht.
- Die anderen Modelle dienen nur dem Vergleich der Kurvenform (Leitfrage 6), sie werden nicht validiert.

In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from awp2.config import (
    FIT_TIME_COL,
    LOGISTIC_REGRESSION_MAX_ITER,
    MLFLOW_REDUCTION_EXPERIMENT,
    MODEL_DOCS_FIGURES_DIR,
    N_COMPONENTS_COL,
    N_INPUT_FEATURES_COL,
    PCA_STUDY_COMPONENTS,
    PCA_STUDY_SENTINEL2_COMPONENTS,
    PREDICT_TIME_COL,
    REDUCTION_BEST_PARAMS_COL,
    REDUCTION_TUNING_RUN_COL,
    REDUCTION_VARIANT_COL,
    REGION_GROUPING,
    SEED,
    SENTINEL2_SENSOR,
    STD_SUFFIX,
)
from awp2.data import combined_label, load_split
from awp2.experiment import RunConfig, TuneConfig, run
from awp2.explain import feature_groups
from awp2.models import CombinedLabelClassifier, make_combined_svm, make_hierarchical_svm
from awp2.plots import (
    plot_fold_timings,
    plot_metric_comparison,
    plot_pca_curve,
    plot_pca_loadings,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig
from awp2.reduction import (
    ReductionVariant,
    explained_variance_at,
    pca_curve,
    pca_recommendation,
    smallest_equivalent,
    spectral_pca,
)

## 1. Modelle, Raster und Referenzen

- **Kombinierte RBF-SVM** (Referenz des Projekts) und **hierarchische SVM** (Kultur → Stadium je Kultur, `C` fest wie in #163, `gamma` je Ebene gesucht).
- **Logistische Regression** (linear) und **kNN** (abstandsbasiert): Beide sollten am ehesten von PCA profitieren (#206: Merkmalsnutzen ist modellabhängig). kNN kennt keine Klassengewichte; `weights="distance"` und die Balanced Accuracy als Ziel mildern die Unwucht.
- Referenzen ohne PCA: alle 131 Hyperion-Bänder bzw. die 10 simulierten Sentinel-2-Bänder, jeweils mit AEZ und zyklischem Monat.

In [ ]:
base = PreprocessingConfig(use_spectral_standard_scale=True)
all_bands = ReductionVariant(name="all_bands", family="all_bands", preprocessing=base)
sentinel2 = ReductionVariant(
    name="sentinel2",
    family="sentinel2",
    preprocessing=base.model_copy(update={"spectral_sensor": SENTINEL2_SENSOR}),
)
GAMMAS = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
models: dict[str, tuple[str, BaseEstimator, dict[str, list[Any]]]] = {
    "Kombinierte SVM": (
        "combined",
        make_combined_svm(),
        {"estimator__C": [30.0, 100.0, 300.0, 1000.0], "estimator__gamma": GAMMAS},
    ),
    "Hierarchische SVM": (
        "hierarchical",
        make_hierarchical_svm(),
        {
            "crop_estimator__C": [30.0],
            "crop_estimator__gamma": GAMMAS[1:5],
            "stage_estimator__C": [100.0],
            "stage_estimator__gamma": GAMMAS[:5],
        },
    ),
    "Logistische Regression": (
        "combined",
        CombinedLabelClassifier(
            LogisticRegression(
                class_weight="balanced",
                max_iter=LOGISTIC_REGRESSION_MAX_ITER,
                random_state=SEED,
            )
        ),
        {"estimator__C": [1.0, 10.0, 100.0]},
    ),
    "kNN": (
        "combined",
        CombinedLabelClassifier(KNeighborsClassifier(weights="distance")),
        {"estimator__n_neighbors": [1, 3, 5, 10, 20]},
    ),
}
slugs = {
    "Kombinierte SVM": "svm",
    "Hierarchische SVM": "hier_svm",
    "Logistische Regression": "logreg",
    "kNN": "knn",
}


def template(label: str, reference: ReductionVariant) -> TuneConfig:
    approach, _, grid = models[label]
    searched = ", ".join(sorted({key.split("__")[-1] for key in grid}))
    return TuneConfig(
        name=f"pca_study_{slugs[label]}",
        approach=approach,
        experiment=MLFLOW_REDUCTION_EXPERIMENT,
        description=f"PCA-Studie (#213): {label}, {reference.name}, je k gesucht: {searched}",
        param_grid=grid,
        preprocessing=reference.preprocessing,
    )

## 2. Kurven: CV-BAcc je Anzahl Komponenten

`pca_curve` tunt die Referenz und jede PCA-Variante (k = 1 … 80) und bewertet die beste Einstellung je Fold für Kultur, Stadium und kombiniert. Dabei werden Fit- und Vorhersagezeit je Fold gemessen.

In [ ]:
curves = {
    label: pca_curve(model, all_bands, PCA_STUDY_COMPONENTS, template(label, all_bands))
    for label, (_, model, _) in models.items()
}
scores = pd.DataFrame(
    {label: curve.set_index(N_COMPONENTS_COL)["bacc_combined"] for label, curve in curves.items()}
)
scores.round(3)

In [ ]:
pd.DataFrame(
    {
        label: curve.set_index(N_COMPONENTS_COL)[REDUCTION_BEST_PARAMS_COL]
        for label, curve in curves.items()
    }
)

## 3. Knie der Kurve und erklärte Varianz

Die erklärte Varianz stammt aus einer PCA auf dem ganzen Trainingsteil (nur zur Beschreibung, nicht fürs Modell).

In [ ]:
hyperion_pca = spectral_pca(base)
svm_curve = curves["Kombinierte SVM"]
fig, ax = plt.subplots(figsize=(10, 5))
plot_pca_curve({"Kombinierte SVM": svm_curve}, hyperion_pca.explained_variance_ratio, ax=ax)
ax.set_title("Kombinierte SVM: Qualität und erklärte Varianz je Anzahl Komponenten")
save_doc_figure(fig, "pca_studie_kurve", MODEL_DOCS_FIGURES_DIR)

In [ ]:
knees = {}
for label, curve in curves.items():
    reference = curve.iloc[0]
    knee = smallest_equivalent(curve, all_bands.name)
    knees[label] = {
        "CV Referenz": reference["bacc_combined"],
        "Knie k": None if knee is None else knee[N_COMPONENTS_COL],
        "CV Knie": None if knee is None else knee["bacc_combined"],
        "bestes k": curve.iloc[1:].set_index(N_COMPONENTS_COL)["bacc_combined"].idxmax(),
        "CV bestes k": curve.iloc[1:]["bacc_combined"].max(),
    }
pd.DataFrame(knees).T

In [ ]:
svm_table = svm_curve.set_index(N_COMPONENTS_COL)[
    ["bacc_crop", "bacc_stage", "bacc_combined", f"bacc_combined{STD_SUFFIX}"]
]
svm_table["erklärte Varianz"] = explained_variance_at(
    svm_curve, hyperion_pca.explained_variance_ratio
)
svm_table.round(3)

**Befund Knie (Leitfrage 1):**

- **Kombinierte SVM:** Die Regel trifft k = 8 (0,851, −0,008). Die Kurve ist ab 8 aber flach und verrauscht: k = 10 liegt mit 0,846 knapp außerhalb, k = 15/20 bei 0,850/0,851. Innerhalb von ±0,005 zur Referenz (0,859) liegt PCA erst ab **30 Komponenten** (0,856).
- **Darunter fällt die Kurve steil:** k = 5 → 0,815, k = 3 → 0,787, k = 2 → 0,766, k = 1 → 0,696.
- **Gamma hängt nicht von k ab:** Die beste SVM nutzt bei fast jedem k `gamma = 0,003` wie die Referenz. PCA ohne Whitening erhält die Abstände (die ersten Komponenten tragen fast die ganze Varianz), deshalb passt derselbe Kernel.
- Kultur und Stadium verlieren gleich: Bei k = 8 sinkt die Kultur um 0,010, das Stadium um 0,004.

### Varianz ist nicht Trennkraft

Wie viel trennt jede Komponente allein? Der F-Wert (ANOVA) jeder Komponente gegen die 23 kombinierten Klassen auf dem Trainingsteil, neben ihrem Varianzanteil.

In [ ]:
y_train = load_split().y_train
f_values, _ = f_classif(hyperion_pca.scores, combined_label(y_train))
separation = pd.DataFrame(
    {
        "Varianzanteil": hyperion_pca.explained_variance_ratio,
        "F-Wert": f_values,
    }
).head(30)
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
separation["Varianzanteil"].mul(100).plot.bar(ax=axes[0], color="grey")
axes[0].set_ylabel("Varianzanteil [%]")
axes[0].set_yscale("log")
separation["F-Wert"].plot.bar(ax=axes[1], color="C0")
axes[1].set_ylabel("F-Wert (23 Klassen)")
axes[1].set_yscale("log")
fig.suptitle("Varianz vs. Trennkraft der ersten 30 Komponenten (Trainingsteil)")
fig.tight_layout()
save_doc_figure(fig, "pca_studie_trennkraft", MODEL_DOCS_FIGURES_DIR)
separation.assign(Rang_F=separation["F-Wert"].rank(ascending=False)).head(12).round(3)

**Befund Varianz (Leitfrage 2):**

- **2 Komponenten erklären 91,8 % der Varianz, erreichen aber nur 0,766** (−0,093). 8 Komponenten erklären 99,6 % – erst dann ist die Qualität erreicht. Die letzten 0,4 % der Varianz tragen also die Hälfte der Trennkraft.
- **Die Rangfolge der Trennkraft folgt nicht der Varianz:** PC2 (29 % der Varianz) liegt beim F-Wert nur auf Rang 7. PC3 (4,6 %) ist Rang 2, PC6 (0,7 %) Rang 3, PC11 (< 0,1 %) Rang 6.
- PCA sortiert nach Varianz, nicht nach Klassen. Die großen Komponenten beschreiben Helligkeit und Boden/Vegetation, die alle Klassen gemeinsam haben.

## 4. Modellabhängigkeit

Differenz zur eigenen Referenz ohne PCA, damit die Modelle trotz unterschiedlicher Qualität auf einer Skala liegen. Gepunktet: die Verlustgrenze −0,01 der Knie-Regel.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_pca_curve(curves, relative=True, ax=ax)
ax.set_title("PCA je Modell: Abstand zur Referenz ohne PCA")
ax.set_ylim(-0.25, 0.05)
save_doc_figure(fig, "pca_studie_modelle", MODEL_DOCS_FIGURES_DIR)

**Befund Modellabhängigkeit (Leitfrage 6):**

| Modell | ohne PCA | Knie (Regel) | bestes k |
| --- | --- | --- | --- |
| Kombinierte SVM | 0,859 | k = 8: 0,851 | 60: 0,860 |
| Hierarchische SVM | 0,846 | k = 15: 0,839 | 60: 0,847 |
| Logistische Regression | 0,808 | k = 8: 0,800 | **20: 0,820 (+0,012)** |
| kNN (k = 1) | 0,719 | k = 10: 0,711 | 30: 0,720 |

- **Die Form ist für alle Modelle gleich:** Bis etwa 5 Komponenten ein steiler Abfall, ab 8–15 im Rauschen der Referenz.
- **Nur die logistische Regression gewinnt:** PCA-20 liegt +0,012 über allen Bändern (≥ `MIN_CV_GAIN`). Die PCA entfernt dort die stark korrelierten Bänder, an denen das lineare Modell überanpasst. Sie bleibt aber 0,04 unter der SVM.
- **kNN profitiert nicht**, obwohl er abstandsbasiert ist: Die Abstände bleiben unter PCA praktisch gleich (s. o.), und 1-NN bleibt weit hinter der SVM.
- Die hierarchische SVM braucht etwas mehr Komponenten (15) als die kombinierte (8).
- Alle Werte sind auf denselben Folds getunt und bewertet, also leicht optimistisch. „Bestes k = 60“ (+0,001) ist kein Gewinn.

## 5. Praktischer Gewinn: Merkmale und Zeit

Mittel über die fünf Folds: Fit = Vorverarbeitung + Training auf vier Folds (≈ 3 580 Zeilen), Vorhersage = Vorverarbeitung + Vorhersage des fünften Folds (≈ 895 Zeilen). Gemessen auf einem Rechner nacheinander, also nur relativ vergleichbar.

In [ ]:
timing = pd.concat(
    {
        label: curve.set_index(N_COMPONENTS_COL)[
            [N_INPUT_FEATURES_COL, FIT_TIME_COL, PREDICT_TIME_COL]
        ]
        for label, curve in curves.items()
    },
    axis=1,
)
timing.round(3)

In [ ]:
fig = plot_fold_timings(curves)
save_doc_figure(fig, "pca_studie_zeit", MODEL_DOCS_FIGURES_DIR)

In [ ]:
total_time = {
    label: curve.set_index(N_COMPONENTS_COL)[[FIT_TIME_COL, PREDICT_TIME_COL]].sum(axis=1)
    for label, curve in curves.items()
}
speedup = pd.DataFrame({label: times.iloc[0] / times for label, times in total_time.items()})
speedup.round(2)

**Befund Zeit (Leitfrage 3):**

- **Merkmale:** Das Knie der SVM (k = 8) gibt dem Modell 17 statt 140 Merkmale (−88 %). **Gemessen werden weiter alle 131 Bänder.**
- **Zeit bei der SVM kaum geringer:** Training ≈ 0,21 statt 0,22 s je Fold, Vorhersage ≈ 0,07 statt 0,14 s – zusammen nur Faktor ≈ 1,3. Die SVM-Kosten hängen vor allem an der Zahl der Zeilen und Stützvektoren, nicht an den Merkmalen; bei k ≤ 3 dauert das Training sogar länger (mehr Stützvektoren). Bei unter einer halben Sekunde je Fold ist der Gewinn praktisch bedeutungslos.
- **Logistische Regression:** Hier spart PCA wirklich Zeit (≈ 19 s → 1,6–2,5 s je Fold, Faktor 7–13), weil lbfgs auf korrelierten Bändern langsam konvergiert. Über 40 Komponenten wird sie wieder langsamer als ohne PCA.
- **kNN wird mit PCA langsamer** (Vorhersage ≈ 0,08 statt 0,02 s ab k = 8), vermutlich wählt scikit-learn bei wenigen Merkmalen eine Baumsuche, die hier langsamer ist.
- Zeiten aus einem Lauf auf einem Rechner, ohne Wiederholung – nur als Größenordnung lesen.

## 6. Was steckt in den Komponenten?

Ladungen der ersten vier Komponenten über der Wellenlänge (PCA auf dem Trainingsteil, standardisierte Bänder). Lücken sind entfernte Wasserdampf-Bänder.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
plot_pca_loadings(hyperion_pca.loadings, hyperion_pca.explained_variance_ratio, ax=ax)
ax.set_title("Ladungen PC1–PC4 (Hyperion, standardisiert)", pad=18)
save_doc_figure(fig, "pca_studie_ladungen", MODEL_DOCS_FIGURES_DIR)

In [ ]:
regions = feature_groups(list(hyperion_pca.loadings.index), REGION_GROUPING)
pd.DataFrame(
    {
        region: hyperion_pca.loadings.loc[bands, ["PC1", "PC2", "PC3", "PC4"]].mean()
        for region, bands in regions.items()
    }
).T.round(3)

**Befund Ladungen (Leitfrage 4):**

| Komponente | Varianz | Muster | Deutung |
| --- | --- | --- | --- |
| PC1 | 62,6 % | VIS, SWIR1, SWIR2 positiv, NIR negativ | Boden/trockene Pflanze gegen grüne Biomasse (umgekehrter NDVI-Kontrast) |
| PC2 | 29,2 % | überall positiv, am stärksten im NIR | Helligkeit, vor allem NIR-Plateau (Blattstruktur, Biomasse) |
| PC3 | 4,6 % | VIS stark positiv, SWIR negativ | Pigmente gegen Wasser/Lignin – Kontrast sichtbar zu SWIR |
| PC4 | 1,1 % | Spitze im Red Edge (≈ 712–722 nm), Blau negativ, Flanken der Wasserbanden bei 1 150–1 320 nm | Lage der Red Edge und Flanken der Wasserabsorption |

- **Bezug zur Feature Importance:** Das wichtigste Einzelband der SVM ist `X722` (Red Edge), gefolgt von `X1316` am Rand einer Wasserbande ([Feature Importance](../docs/modelle/feature-importance.md)). Genau diese Stellen stecken erst in PC4 mit 1,1 % Varianz. Die trennende Information liegt also in kleinen Komponenten – darum braucht PCA mindestens 8 davon.
- PC1/PC2 entsprechen den Merkmalen `PC_1`/`PC_2` aus der EDA; als Zusatzmerkmale brachten sie nichts (#155).

## 7. PCA auf Sentinel-2: Reduktion plus Messersparnis

Sentinel-2 spart bereits Messungen (10 statt 131 Bänder). Bringt eine zusätzliche PCA (10 → k) noch etwas?

In [ ]:
s2_models = ["Kombinierte SVM", "Hierarchische SVM"]
s2_curves = {
    label: pca_curve(
        models[label][1],
        sentinel2,
        PCA_STUDY_SENTINEL2_COMPONENTS,
        template(label, sentinel2),
    )
    for label in s2_models
}
s2_pca = spectral_pca(sentinel2.preprocessing)
s2_table = pd.concat(
    {
        label: curve.set_index(N_COMPONENTS_COL)[
            ["bacc_combined", f"bacc_combined{STD_SUFFIX}", FIT_TIME_COL, PREDICT_TIME_COL]
        ]
        for label, curve in s2_curves.items()
    },
    axis=1,
)
s2_table["erklärte Varianz"] = explained_variance_at(
    s2_curves["Kombinierte SVM"], s2_pca.explained_variance_ratio
)
s2_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_pca_curve(s2_curves, s2_pca.explained_variance_ratio, ax=ax)
ax.set_title("Sentinel-2: PCA auf 10 Bändern")
save_doc_figure(fig, "pca_studie_sentinel2", MODEL_DOCS_FIGURES_DIR)
pd.DataFrame(
    {label: pca_recommendation(curve).model_dump() for label, curve in s2_curves.items()}
).T

**Befund Sentinel-2 (Leitfrage 5):**

- **PCA auf Sentinel-2 lohnt sich nicht.** 10 → 8 Komponenten hält die Qualität (SVM 0,859 statt 0,860, hierarchisch 0,846 = 0,846), 10 → 5 kostet schon 0,024 bzw. 0,023.
- Es bleiben 10 gemessene Bänder; die Zeit ändert sich nicht (Faktor ≈ 1,0). Die Messersparnis kommt allein vom Sensor, nicht von der PCA.
- **Nebenbefund:** Die hierarchische SVM erreicht auf Sentinel-2 mit dem breiteren `gamma`-Raster (Kultur 0,1, Stadium 0,03) CV 0,846, in #163 nur 0,834 – gleichauf mit allen Bändern. Nicht validiert.

## 8. Empfehlung nach Regel und Validierung

Die Regel aus der Einleitung, angewandt auf die kombinierte SVM (Hyperion und Sentinel-2).

In [ ]:
candidates = {
    all_bands.name: (all_bands, curves["Kombinierte SVM"]),
    sentinel2.name: (sentinel2, s2_curves["Kombinierte SVM"]),
}
decisions = {name: pca_recommendation(curve) for name, (_, curve) in candidates.items()}
pd.DataFrame({name: decision.model_dump() for name, decision in decisions.items()}).T

In [ ]:
recommended = [name for name, decision in decisions.items() if decision.recommended]
if recommended:
    reference, curve = candidates[recommended[0]]
    row = curve.set_index(REDUCTION_VARIANT_COL).loc[decisions[recommended[0]].knee]
    result = run(
        make_combined_svm().set_params(**row[REDUCTION_BEST_PARAMS_COL]),
        RunConfig(
            name=f"pca_study_svm_{row.name}",
            approach="combined",
            experiment=MLFLOW_REDUCTION_EXPERIMENT,
            description=f"PCA-Studie (#213): Knie {row.name} per Regel empfohlen",
            tuning_run=row[REDUCTION_TUNING_RUN_COL],
            preprocessing=reference.preprocessing.model_copy(
                update={"spectral_pca_components": int(row[N_COMPONENTS_COL])}
            ),
        ),
    )
    display(pd.Series({**result.metrics.model_dump(), "run_id": result.run_id}))
    plot_metric_comparison({row.name: result.metrics})
else:
    print("Keine PCA-Variante erfüllt die Regel – keine Validierung.")

**Entscheidung:** Kein Knie erfüllt die Regel (Hyperion: −0,008 CV, Zeitfaktor ≈ 1,3; Sentinel-2: −0,001, ≈ 1,0). Es gibt **keine Validierung**.

## Fazit

| Frage | Antwort |
| --- | --- |
| Ab wie vielen Komponenten? | Regel: 8 (−0,008); stabil im Rauschen (±0,005) ab 30. Unter 8 steiler Abfall. |
| Erklärte Varianz? | 2 PCs = 91,8 % Varianz, aber −0,093 BAcc. Erst 99,6 % (8 PCs) reichen. Varianz ≠ Trennkraft. |
| Praktischer Gewinn? | Merkmale 140 → 17, Messungen bleiben 131. SVM-Zeit nur Faktor ≈ 1,3 bei < 0,5 s je Fold. |
| Inhalt der Komponenten? | PC1 Boden vs. Biomasse, PC2 Helligkeit (NIR), PC3 VIS vs. SWIR, PC4 Red Edge und Wasserflanken. |
| Sentinel-2 + PCA? | 10 → 8 ohne Verlust, aber ohne Nutzen; 10 → 5 kostet 0,024. |
| Modellabhängig? | Form gleich; nur die logistische Regression gewinnt (+0,012, 7–13× schneller), bleibt aber 0,04 unter der SVM. |

- **Für M3 ist PCA nicht der richtige Hebel:** Sie verkleinert das Modell, aber nicht den Messaufwand, und die SVM wird kaum schneller. Den Messaufwand senkt die Sensorwahl (Sentinel-2, 92 % weniger Bänder).
- **Die trennende Information steckt in kleinen Komponenten** (Red Edge, Wasserflanken). Darum versagen Reduktionen nach Varianz bei wenigen Komponenten.
- **Einschränkungen:** Referenz ist die Vorverarbeitung aus #155 (AEZ + zyklischer Monat, CV 0,859), nicht das aktuell beste Modell mit AEZ × Monat (0,864). kNN ohne Klassengewichte. Zeiten aus einem Lauf.
- **Offen:** logistische Regression mit PCA-20 als schnelles lineares Modell (z. B. für Ensembles), hierarchische SVM auf Sentinel-2 mit breitem `gamma`-Raster validieren.